# WaterNet v2 — Altitude Estimation from Nadir Water Images

**Paper:** *Deep Learning-Based Altitude Estimation for Unmanned Aerial Vehicles over Water Surfaces*

## What this notebook teaches

| Phase | Topic | Key insight |
|-------|-------|-------------|
| 0 | HSV Value channel vs Grayscale | `V = max(R,G,B)` preserves specular peaks |
| 2a | FFT frequency as altitude proxy | High-freq energy ↓ as altitude ↑ |
| 2b | Shi-Tomasi corner density | Glint count ↓ monotonically with altitude |
| 2c | 12-element feature vector | Handcrafted features encoding all above signals |

> **How to run:** `Runtime → Run all`  (Google Colab)  
> No external dataset required — all images are generated synthetically inside the notebook.

In [ ]:
# ── Install dependencies (uncomment on Google Colab) ──────────────────────────
# !pip install -q opencv-python-headless scipy scikit-learn pyyaml matplotlib seaborn

import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy import stats
import yaml

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ── Optional TensorFlow (used from Cell 8 onwards) ─────────────────────────────
try:
    import tensorflow as tf
    tf.random.set_seed(SEED)
    print(f"TensorFlow {tf.__version__} loaded.")
except ImportError:
    print("TensorFlow not found — only Cells 0-7 are active.")

print(f"NumPy  {np.__version__}")
print(f"OpenCV {cv2.__version__}")
print("Environment ready ✓")


In [ ]:
# ── Configuration loader ───────────────────────────────────────────────────────
# Ported from waternet_v2/configs/__init__.py
# Assumes waternet_v2/configs/default.yaml is present in the working directory.

CONFIG_PATH = "waternet_v2/configs/default.yaml"


def _deep_merge(base: dict, override: dict) -> None:
    """Recursively merge *override* into *base* in-place."""
    for key, value in override.items():
        if key in base and isinstance(base[key], dict) and isinstance(value, dict):
            _deep_merge(base[key], value)
        else:
            base[key] = value


def load_config(path: str = CONFIG_PATH, override_path: str = None) -> dict:
    """Load YAML configuration with optional deep-merge override.

    Args:
        path: Path to the base YAML config (default.yaml).
        override_path: Optional path to a custom YAML file whose values
                       override the base config.

    Returns:
        Nested dict of all hyperparameters.
    """
    with open(path) as fh:
        config = yaml.safe_load(fh)
    if override_path is not None:
        with open(override_path) as fh:
            overrides = yaml.safe_load(fh) or {}
        _deep_merge(config, overrides)
    return config


cfg = load_config()

# ── Pretty-print ───────────────────────────────────────────────────────────────
for section, values in cfg.items():
    print(f"\n[{section}]")
    if isinstance(values, dict):
        for k, v in values.items():
            print(f"  {k}: {v}")
    else:
        print(f"  {values}")


In [ ]:
# ── Problem Setup ──────────────────────────────────────────────────────────────
"""
Task:  Estimate UAV altitude (50 – 800 cm) from a single nadir RGB image.

Physics model:
  • Low altitude  → camera resolves individual capillary waves
                  → high spatial frequency + dense specular glints
  • High altitude → surface appears smooth, waves average out
                  → low spatial frequency + sparse glints

Key constraint (Nadir assumption):
  The camera always points straight down, so the image has no preferred
  orientation → full 360° rotation augmentation is valid.
"""

ALT_MIN_CM = cfg["data"]["alt_min_cm"]   # 50
ALT_MAX_CM = cfg["data"]["alt_max_cm"]   # 800
IMG_H, IMG_W = cfg["data"]["image_size"]  # 224, 224

PROBE_ALTS = [50, 100, 200, 300, 400, 500, 600, 700, 800]


def generate_synthetic_water_image(
    altitude_cm: float,
    size: int = 224,
    seed: int = None,
) -> np.ndarray:
    """Generate a synthetic nadir water image at the given altitude.

    Physics model:
      - Sinusoidal ripples whose spatial frequency scales with
        (1 - alt_norm): denser at low altitude.
      - Gaussian noise proportional to texture variance.
      - Bright disk-shaped specular glints whose count scales with
        (1 - alt_norm)^2: many close up, few at distance.

    Args:
        altitude_cm: UAV altitude above water [50, 800].
        size:        Output image side length in pixels.
        seed:        Optional RNG seed for reproducibility.

    Returns:
        RGB uint8 NumPy array of shape (size, size, 3).
    """
    rng = np.random.RandomState(
        seed if seed is not None else int(altitude_cm) % (2**31)
    )
    alt_norm = float(np.clip((altitude_cm - ALT_MIN_CM) / (ALT_MAX_CM - ALT_MIN_CM),
                             0.0, 1.0))

    # ── Ripple texture ──────────────────────────────────────────────────────
    n_waves   = max(2, int(12 * (1.0 - alt_norm) + 2))
    freq_base = 3.0 + 18.0 * (1.0 - alt_norm)
    t         = np.linspace(0, 2 * np.pi, size, endpoint=False)
    xx, yy    = np.meshgrid(t, t)
    ripple    = np.zeros((size, size), dtype=np.float32)
    for _ in range(n_waves):
        angle = rng.uniform(0, np.pi)
        phase = rng.uniform(0, 2 * np.pi)
        freq  = rng.uniform(0.8, 1.2) * freq_base
        ripple += np.sin(freq * (xx * np.cos(angle) + yy * np.sin(angle)) + phase)
    ripple /= n_waves
    ripple = (ripple - ripple.min()) / (ripple.max() - ripple.min() + 1e-8)

    # ── Sensor noise ────────────────────────────────────────────────────────
    noise_std = 0.08 * (1.0 - alt_norm) + 0.02
    noise = rng.normal(0, noise_std, (size, size)).astype(np.float32)
    texture = np.clip(ripple * 0.75 + noise * 0.25, 0.0, 1.0)

    # ── Specular glints ─────────────────────────────────────────────────────
    n_glints = int(60 * (1.0 - alt_norm) ** 2) + 1
    for _ in range(n_glints):
        cx = rng.randint(8, size - 8)
        cy = rng.randint(8, size - 8)
        r  = rng.randint(2, 5)
        brightness = rng.uniform(0.85, 1.0)
        yy_g, xx_g = np.ogrid[-cy:size - cy, -cx:size - cx]
        mask = (xx_g ** 2 + yy_g ** 2) <= r ** 2
        texture[mask] = np.maximum(texture[mask], brightness)

    # ── Blue-teal water palette ──────────────────────────────────────────────
    r_ch = np.clip(0.05 + texture * 0.35, 0, 1)
    g_ch = np.clip(0.20 + texture * 0.30, 0, 1)
    b_ch = np.clip(0.40 + texture * 0.25, 0, 1)
    return (np.stack([r_ch, g_ch, b_ch], axis=-1) * 255).astype(np.uint8)


# ── Quick visual check ─────────────────────────────────────────────────────────
sample_alts = [50, 200, 500, 800]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, alt in zip(axes, sample_alts):
    ax.imshow(generate_synthetic_water_image(alt, seed=0))
    ax.set_title(f"altitude = {alt} cm", fontsize=11)
    ax.axis("off")
fig.suptitle("Synthetic Water Images at Different Altitudes", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("Observation: at 50 cm — dense ripples + bright glints; "
      "at 800 cm — smooth, uniform surface.")


In [ ]:
# ── Phase 0: HSV Value Channel vs Grayscale ────────────────────────────────────
# Ported from waternet_v2/data/preprocessing.py
"""
Paper Section 2.3.1

HSV Value channel:  V = max(R, G, B)       [Eq. 2.11]
ITU-R BT.601:       Y = 0.299R + 0.587G + 0.114B

The weighted average in BT.601 ATTENUATES specular intensity peaks because it
blends the bright channel with dimmer ones.  V = max() PRESERVES those peaks
— which is where altitude information is encoded (glint density, wave contrast).
"""


def extract_value_channel(
    img_rgb: np.ndarray,
    target_size: tuple = (224, 224),
) -> np.ndarray:
    """Resize and return the normalised HSV V channel.  V = max(R, G, B).

    Args:
        img_rgb:     Input image, shape (H, W, 3), uint8 RGB.
        target_size: (width, height) resize target.

    Returns:
        V channel, float32 in [0, 1], shape (H, W).
    """
    if img_rgb.shape[:2] != (target_size[1], target_size[0]):
        img_rgb = cv2.resize(img_rgb, target_size, interpolation=cv2.INTER_AREA)
    hsv = cv2.cvtColor(img_rgb.astype(np.uint8), cv2.COLOR_RGB2HSV)
    return hsv[:, :, 2].astype(np.float32) / 255.0


def grayscale_from_rgb(img_rgb: np.ndarray) -> np.ndarray:
    """ITU-R BT.601 luminance conversion (comparison baseline).

    Y = 0.299R + 0.587G + 0.114B

    Args:
        img_rgb: Input image, shape (H, W, 3), uint8 or float32.

    Returns:
        Grayscale image, float32 in [0, 1].
    """
    img = img_rgb.astype(np.float32) / 255.0 if img_rgb.dtype == np.uint8 else img_rgb
    return (0.299 * img[:, :, 0]
            + 0.587 * img[:, :, 1]
            + 0.114 * img[:, :, 2]).astype(np.float32)


# ── Side-by-side visual comparison ─────────────────────────────────────────────
demo_alt = 100
img_rgb  = generate_synthetic_water_image(demo_alt, seed=7)
v_chan   = extract_value_channel(img_rgb)
gs_chan  = grayscale_from_rgb(img_rgb)
diff     = v_chan - gs_chan

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img_rgb);               axes[0].set_title("RGB (input)")
axes[1].imshow(v_chan,  cmap="gray");  axes[1].set_title("HSV V channel\n[this work]")
axes[2].imshow(gs_chan, cmap="gray");  axes[2].set_title("Grayscale BT.601\n[baseline]")
im = axes[3].imshow(diff, cmap="RdBu_r", vmin=-0.3, vmax=0.3)
axes[3].set_title("Difference  V − Y\n(red = V brighter)")
plt.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04)
for ax in axes:
    ax.axis("off")
plt.suptitle(f"HSV V channel vs Grayscale  (altitude = {demo_alt} cm)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Quantitative comparison across altitudes ────────────────────────────────────
print(f"{'Alt':>6}  {'V mean':>8}  {'GS mean':>8}  {'V max':>8}  {'GS max':>8}  {'V−GS max':>10}")
print("-" * 58)
for alt in [50, 100, 200, 400, 800]:
    img = generate_synthetic_water_image(alt, seed=0)
    v   = extract_value_channel(img)
    gs  = grayscale_from_rgb(img)
    print(f"{alt:>4} cm  {v.mean():>8.4f}  {gs.mean():>8.4f}  "
          f"{v.max():>8.4f}  {gs.max():>8.4f}  {(v - gs).max():>10.4f}")

print("\nKey observation: V max stays closer to 1.0 (specular glints preserved);")
print("                 GS max is lower because weighted averaging dilutes them.")


In [ ]:
# ── Phase 2a: 2D FFT Frequency as Altitude Proxy ─────────────────────────────
# Ported from waternet_v2/data/preprocessing.py
"""
Paper Section 2.3.2

F(u,v) = Σ_x Σ_y f(x,y) · exp(−j2π(ux/M + vy/N))   [Eq. 2.12]

Energy is partitioned by normalised radial distance r ∈ [0, 1] from DC:
  low  band  (r < 0.20) : large-scale waves / smooth surface → ↑ at high altitude
  mid  band  (0.20–0.60): medium texture
  high band  (r ≥ 0.60) : fine ripples / capillary waves    → ↑ at low altitude
"""


def compute_fft_magnitude(v_channel: np.ndarray) -> np.ndarray:
    """2D FFT log-magnitude spectrum, normalised to [0, 1].

    Args:
        v_channel: Single-channel float32 image in [0, 1].

    Returns:
        Log-magnitude spectrum, float32 in [0, 1], same spatial shape.
    """
    f_shift   = np.fft.fftshift(np.fft.fft2(v_channel))
    magnitude = np.log1p(np.abs(f_shift)).astype(np.float32)
    max_val   = magnitude.max()
    if max_val > 0:
        magnitude /= max_val
    return magnitude


def compute_fft_energy_bands(
    v_channel: np.ndarray,
    low_cutoff: float = 0.20,
    mid_cutoff: float = 0.60,
) -> tuple:
    """Partition FFT energy into (low, mid, high) radial frequency bands.

    Args:
        v_channel:   Single-channel float32 image in [0, 1].
        low_cutoff:  Normalised radial boundary separating low from mid band.
        mid_cutoff:  Normalised radial boundary separating mid from high band.

    Returns:
        (energy_low, energy_mid, energy_high) — fractions summing to ~1.
    """
    h, w = v_channel.shape
    F    = np.fft.fftshift(np.fft.fft2(v_channel))
    mag  = np.abs(F)

    cy, cx = h // 2, w // 2
    yy, xx = np.mgrid[0:h, 0:w]
    r_norm = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2) / (
        np.sqrt(cy ** 2 + cx ** 2) + 1e-10
    )

    total  = mag.sum() + 1e-10
    e_low  = float(mag[r_norm <  low_cutoff].sum() / total)
    e_mid  = float(mag[(r_norm >= low_cutoff) & (r_norm < mid_cutoff)].sum() / total)
    e_high = float(mag[r_norm >= mid_cutoff].sum() / total)
    return e_low, e_mid, e_high


# ── Verify FFT energy shifts with altitude ──────────────────────────────────────
N_SAMPLES = 8

records = []
print(f"{'Alt (cm)':>10}  {'Low band':>10}  {'Mid band':>10}  {'High band':>10}")
print("-" * 50)
for alt in PROBE_ALTS:
    lows, mids, highs = [], [], []
    for s in range(N_SAMPLES):
        v = extract_value_channel(generate_synthetic_water_image(alt, seed=s))
        l, m, h_e = compute_fft_energy_bands(v)
        lows.append(l); mids.append(m); highs.append(h_e)
    records.append((alt, np.mean(lows), np.mean(mids), np.mean(highs)))
    print(f"{alt:>8} cm  {np.mean(lows):>10.4f}  {np.mean(mids):>10.4f}  {np.mean(highs):>10.4f}")

print("\nExpected: high-band energy DECREASES as altitude INCREASES.")

# ── Plot ────────────────────────────────────────────────────────────────────────
alts_r  = np.array([r[0] for r in records])
lows_r  = np.array([r[1] for r in records])
mids_r  = np.array([r[2] for r in records])
highs_r = np.array([r[3] for r in records])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(alts_r, lows_r,  "o-", label="Low  (r < 0.20)",   color="#1f77b4", lw=2)
axes[0].plot(alts_r, mids_r,  "s-", label="Mid  (0.20 – 0.60)", color="#ff7f0e", lw=2)
axes[0].plot(alts_r, highs_r, "^-", label="High (r ≥ 0.60)",   color="#d62728", lw=2)
axes[0].set_xlabel("Altitude (cm)"); axes[0].set_ylabel("Fractional Energy")
axes[0].set_title("FFT Energy Bands vs Altitude")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

img_lo  = generate_synthetic_water_image(50,  seed=0)
img_hi  = generate_synthetic_water_image(800, seed=0)
spec_lo = compute_fft_magnitude(extract_value_channel(img_lo))
spec_hi = compute_fft_magnitude(extract_value_channel(img_hi))
axes[1].imshow(np.hstack([spec_lo, spec_hi]), cmap="inferno", aspect="auto")
axes[1].axvline(x=spec_lo.shape[1] - 0.5, color="white", linewidth=1.5)
axes[1].set_title("FFT log-magnitude spectrum\n"
                  "50 cm (left)  |  800 cm (right)")
axes[1].set_xlabel("50 cm: energy spread to high freq  |  800 cm: bright DC centre")
axes[1].set_xticks([]); axes[1].set_yticks([])

plt.tight_layout()
plt.show()


In [ ]:
# ── Phase 2b: Shi-Tomasi Corner Density ───────────────────────────────────────
# Ported from waternet_v2/data/preprocessing.py
"""
Paper Section 2.3.3

Shi-Tomasi criterion: corner if  min(λ₁, λ₂) > τ   [Eq. 2.16]
where λ₁, λ₂ are eigenvalues of the local structure tensor M.

Specular reflection peaks create high-gradient regions that the detector
classifies as "corners".  Their count DECREASES with altitude because
fewer individual wave facets produce resolvable specular reflections.
"""


def count_shi_tomasi_features(
    v_channel: np.ndarray,
    max_corners: int = 100,
    quality_level: float = 0.01,
    min_distance: float = 5.0,
) -> int:
    """Count Shi-Tomasi corners as a proxy for specular glint density.

    Args:
        v_channel:     Single-channel float32 image in [0, 1].
        max_corners:   Maximum corners to detect.
        quality_level: Minimum corner quality (fraction of max eigenvalue).
        min_distance:  Minimum pixel distance between returned corners.

    Returns:
        Integer count of detected corners in [0, max_corners].
    """
    img_u8  = (v_channel * 255).astype(np.uint8)
    corners = cv2.goodFeaturesToTrack(
        img_u8,
        maxCorners=max_corners,
        qualityLevel=quality_level,
        minDistance=min_distance,
    )
    return int(len(corners)) if corners is not None else 0


# ── Corner counts vs altitude ───────────────────────────────────────────────────
N_SAMPLES = 10
corner_means, corner_stds = [], []

print(f"{'Alt (cm)':>10}  {'Mean corners':>14}  {'Std':>8}")
print("-" * 38)
for alt in PROBE_ALTS:
    counts = [
        count_shi_tomasi_features(
            extract_value_channel(generate_synthetic_water_image(alt, seed=s))
        )
        for s in range(N_SAMPLES)
    ]
    m, s = np.mean(counts), np.std(counts)
    corner_means.append(m); corner_stds.append(s)
    print(f"{alt:>8} cm  {m:>14.1f}  {s:>8.1f}")

print("\nExpected: corner count DECREASES monotonically with altitude.")

# ── Plot ────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].errorbar(PROBE_ALTS, corner_means, yerr=corner_stds,
                 fmt="o-", capsize=4, color="#2ca02c", linewidth=2, elinewidth=1)
axes[0].set_xlabel("Altitude (cm)"); axes[0].set_ylabel("Corner Count")
axes[0].set_title("Shi-Tomasi Corner Density vs Altitude\n"
                  "(proxy for specular glint density)")
axes[0].grid(True, alpha=0.3)

demo_img  = generate_synthetic_water_image(100, seed=42)
demo_v    = extract_value_channel(demo_img)
demo_u8   = (demo_v * 255).astype(np.uint8)
corners   = cv2.goodFeaturesToTrack(demo_u8, maxCorners=100,
                                    qualityLevel=0.01, minDistance=5)
vis_img   = demo_img.copy()
n_corners = 0
if corners is not None:
    n_corners = len(corners)
    for pt in corners.reshape(-1, 2):
        cv2.circle(vis_img, (int(pt[0]), int(pt[1])), 4, (255, 50, 50), -1)
axes[1].imshow(vis_img)
axes[1].set_title(f"Detected corners at 100 cm: {n_corners}\n"
                  "(red dots = specular glint hotspots)")
axes[1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ── Feature Vector Assembly ────────────────────────────────────────────────────
# Ported from waternet_v2/data/preprocessing.py
"""
The 12-element handcrafted feature vector (Paper Section 2.3):

  Index  Name               Description
  ─────  ─────────────────  ──────────────────────────────────────────────────
  0      mean_v             Mean pixel intensity of the V channel
  1      std_v              Standard deviation of V channel
  2      skew_v             Skewness (asymmetry of intensity distribution)
  3      kurt_v             Kurtosis (tailedness; glints → heavy tail → ↑ kurtosis)
  4      fft_energy_low     FFT energy in r < 0.20 (DC / large-scale)
  5      fft_energy_mid     FFT energy in 0.20 ≤ r < 0.60
  6      fft_energy_high    FFT energy in r ≥ 0.60 (fine ripples)
  7      grad_mean          Mean Sobel gradient magnitude
  8      grad_std           Std of Sobel gradient magnitude
  9      entropy            Normalised image entropy (texture complexity)
  10     shi_tomasi_count   Corner count (glint density proxy)
  11     local_std_mean     Mean of 8×8 block standard deviations
"""

FEATURE_NAMES = [
    "mean_v", "std_v", "skew_v", "kurt_v",
    "fft_energy_low", "fft_energy_mid", "fft_energy_high",
    "grad_mean", "grad_std", "entropy",
    "shi_tomasi_count", "local_std_mean",
]


def extract_feature_vector(v_channel: np.ndarray) -> np.ndarray:
    """Extract the 12-element handcrafted feature vector from a V channel.

    Args:
        v_channel: Single-channel float32 image in [0, 1], shape (H, W).

    Returns:
        Feature vector, float32 array of length 12.
    """
    flat = v_channel.ravel()

    # ── 0–3: Pixel statistics ───────────────────────────────────────────────
    mean_v = float(flat.mean())
    std_v  = float(flat.std())
    skew_v = float(stats.skew(flat))
    kurt_v = float(stats.kurtosis(flat))

    # ── 4–6: FFT energy bands ───────────────────────────────────────────────
    e_low, e_mid, e_high = compute_fft_energy_bands(v_channel)

    # ── 7–8: Sobel gradient statistics ──────────────────────────────────────
    gx       = cv2.Sobel(v_channel, cv2.CV_64F, 1, 0, ksize=3)
    gy       = cv2.Sobel(v_channel, cv2.CV_64F, 0, 1, ksize=3)
    grad_mag  = np.sqrt(gx ** 2 + gy ** 2)
    grad_mean = float(grad_mag.mean())
    grad_std  = float(grad_mag.std())

    # ── 9: Image entropy ─────────────────────────────────────────────────────
    hist, _ = np.histogram(flat, bins=32, range=(0.0, 1.0), density=True)
    hist   += 1e-10
    entropy = float(-np.sum(hist * np.log2(hist)) / np.log2(32))

    # ── 10: Shi-Tomasi corner count ──────────────────────────────────────────
    shi_count = float(count_shi_tomasi_features(v_channel))

    # ── 11: Mean of 8×8 block standard deviations ───────────────────────────
    h, w  = v_channel.shape
    patch = 8
    local_stds = [
        v_channel[i:i + patch, j:j + patch].std()
        for i in range(0, h - patch, patch)
        for j in range(0, w - patch, patch)
    ]
    local_std_mean = float(np.mean(local_stds)) if local_stds else 0.0

    return np.array(
        [mean_v, std_v, skew_v, kurt_v,
         e_low,  e_mid, e_high,
         grad_mean, grad_std, entropy,
         shi_count, local_std_mean],
        dtype=np.float32,
    )


# ── Demo: compare feature vectors at 50 cm vs 800 cm ───────────────────────────
feat_50  = extract_feature_vector(extract_value_channel(
    generate_synthetic_water_image(50,  seed=0)))
feat_800 = extract_feature_vector(extract_value_channel(
    generate_synthetic_water_image(800, seed=0)))

directions = {
    "mean_v":          "— (varies)",
    "std_v":           "↑ at low alt",
    "skew_v":          "— (varies)",
    "kurt_v":          "↑ at low alt (glints)",
    "fft_energy_low":  "↑ at high alt",
    "fft_energy_mid":  "— (varies)",
    "fft_energy_high": "↑ at low alt",
    "grad_mean":       "↑ at low alt",
    "grad_std":        "↑ at low alt",
    "entropy":         "— (varies)",
    "shi_tomasi_count":"↑ at low alt",
    "local_std_mean":  "↑ at low alt",
}

print(f"{'Feature':<22}  {'50 cm':>10}  {'800 cm':>10}  {'Expected trend':>20}")
print("-" * 70)
for name, v50, v800 in zip(FEATURE_NAMES, feat_50, feat_800):
    print(f"{name:<22}  {v50:>10.4f}  {v800:>10.4f}  {directions[name]:>20}")

print(f"\nFeature vector shape: {feat_50.shape}  (12 elements)")

# ── Heatmap: feature values across all probe altitudes ─────────────────────────
feat_matrix = np.array([
    extract_feature_vector(
        extract_value_channel(generate_synthetic_water_image(alt, seed=0))
    )
    for alt in PROBE_ALTS
])

# Normalise each feature column to [0, 1] for visualisation
feat_norm = (feat_matrix - feat_matrix.min(0)) / (feat_matrix.ptp(0) + 1e-8)

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(feat_norm.T, aspect="auto", cmap="viridis", vmin=0, vmax=1)
ax.set_yticks(range(len(FEATURE_NAMES)))
ax.set_yticklabels(FEATURE_NAMES, fontsize=9)
ax.set_xticks(range(len(PROBE_ALTS)))
ax.set_xticklabels([str(a) for a in PROBE_ALTS], fontsize=9)
ax.set_xlabel("Altitude (cm)")
ax.set_title("Feature Vector Heatmap (column-normalised)\n"
             "Rows = features, Columns = altitude levels")
plt.colorbar(im, ax=ax, label="Normalised value [0,1]")
plt.tight_layout()
plt.show()


In [ ]:
# ── Cell 8: Dataset Loader ─────────────────────────────────────────────────────
# Ported from waternet_v2/data/dataset.py
#
# The dataset is assumed to already exist on disk.
# Expected layout:
#   <image_dir>/          ← folder of JPEG/PNG images
#   <csv_path>            ← CSV with columns: nome, distancia
#
# Configured via cfg["data"]["image_dir"] and cfg["data"]["csv_path"].

import pandas as pd
from pathlib import Path


def load_dataset(
    image_dir: str,
    csv_path: str,
    cfg: dict = None,
) -> pd.DataFrame:
    """Load the dataset CSV and validate that every listed image exists on disk.

    Args:
        image_dir: Folder containing the image files.
        csv_path:  CSV file with columns ``nome`` (filename) and
                   ``distancia`` (altitude in cm).
        cfg:       Optional config dict (used only for ``n_bins_stratify``).

    Returns:
        Validated pd.DataFrame with columns ``nome`` and ``distancia``.

    Raises:
        FileNotFoundError: If the CSV or image directory does not exist.
        ValueError:        If required columns are missing from the CSV.
    """
    image_dir = Path(image_dir)
    csv_path  = Path(csv_path)

    if not csv_path.exists():
        raise FileNotFoundError(f"CSV not found: {csv_path}")
    if not image_dir.is_dir():
        raise FileNotFoundError(f"Image directory not found: {image_dir}")

    df = pd.read_csv(csv_path)

    required = {"nome", "distancia"}
    missing_cols = required - set(df.columns)
    if missing_cols:
        raise ValueError(f"CSV is missing columns: {missing_cols}")

    # ── Cross-check files on disk ──────────────────────────────────────────────
    missing_files = df.loc[
        ~df["nome"].apply(lambda n: (image_dir / n).exists()), "nome"
    ]
    if len(missing_files):
        print(f"WARNING: {len(missing_files)} listed file(s) not found on disk "
              f"(first 5: {missing_files.head().tolist()})")
    else:
        print("File check passed: all listed images found on disk.")

    # ── Statistics ─────────────────────────────────────────────────────────────
    print(f"\nDataset loaded from : {csv_path}")
    print(f"  Total samples  : {len(df):,}")
    print(f"  Altitude range : {df['distancia'].min():.1f} – "
          f"{df['distancia'].max():.1f} cm")
    print(f"  Unique images  : {df['nome'].nunique():,}")
    print(f"  Missing files  : {len(missing_files)} / {len(df)}")

    # ── Altitude distribution ──────────────────────────────────────────────────
    n_bins = cfg["data"]["n_bins_stratify"] if cfg else 21
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))

    axes[0].hist(df["distancia"], bins=n_bins,
                 color="#1f77b4", edgecolor="white", linewidth=0.4)
    axes[0].set_xlabel("Altitude (cm)")
    axes[0].set_ylabel("Sample count")
    axes[0].set_title("Altitude Distribution")
    axes[0].grid(True, axis="y", alpha=0.3)

    # Per-altitude bin counts (bar chart)
    bin_counts = df.groupby(
        pd.cut(df["distancia"], bins=n_bins)
    ).size()
    axes[1].bar(range(len(bin_counts)), bin_counts.values,
                color="#ff7f0e", edgecolor="white", linewidth=0.3)
    axes[1].set_xlabel("Altitude bin index")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Samples per Altitude Bin (stratification check)")
    axes[1].grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

    return df


# ── Load ───────────────────────────────────────────────────────────────────────
IMAGE_DIR = cfg["data"]["image_dir"]
CSV_PATH  = cfg["data"]["csv_path"]

df = load_dataset(IMAGE_DIR, CSV_PATH, cfg)
df.head()


In [ ]:
# ── Cell 9: Stratified Split & WaterDataSequence ──────────────────────────────
# Ported from waternet_v2/data/dataset.py
# No augmentation — WaterDataSequence is used with augmenter=None.

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def make_stratified_split(
    df: pd.DataFrame,
    cfg: dict,
) -> tuple:
    """Produce a stratified 70 / 15 / 15 train / val / test split.

    Stratification bins are computed with ``pd.cut`` on the ``distancia``
    column so that every altitude range is proportionally represented in
    all three splits.

    Args:
        df:  DataFrame with a ``distancia`` column.
        cfg: Config dict; reads ``data.test_size``, ``data.val_size``,
             ``data.n_bins_stratify``, ``project.seed``.

    Returns:
        (df_train, df_val, df_test) — each reset-indexed.
    """
    n_bins    = cfg["data"]["n_bins_stratify"]
    test_size = cfg["data"]["test_size"]
    val_size  = cfg["data"]["val_size"]
    seed      = cfg["project"]["seed"]

    strata = pd.cut(df["distancia"], bins=n_bins, labels=False)

    df_train_val, df_test = train_test_split(
        df, test_size=test_size, random_state=seed, stratify=strata,
    )

    strata_tv = pd.cut(df_train_val["distancia"], bins=n_bins, labels=False)
    df_train, df_val = train_test_split(
        df_train_val, test_size=val_size, random_state=seed, stratify=strata_tv,
    )

    print(f"[Split]  train={len(df_train):,}  val={len(df_val):,}  "
          f"test={len(df_test):,}  total={len(df):,}")
    return (
        df_train.reset_index(drop=True),
        df_val.reset_index(drop=True),
        df_test.reset_index(drop=True),
    )


class WaterDataSequence(tf.keras.utils.Sequence):
    """Batch generator: image file → HSV V channel → feature vector → batch.

    Per-sample pipeline:
      1. Read RGB image from ``image_dir / nome``.
      2. Resize and extract HSV V channel (V = max(R,G,B)).
      3. Extract the 12-element handcrafted feature vector.
      4. Yield ``{"image_input": V[..., np.newaxis],
                  "feature_input": feat}`` + pre-scaled target.

    No augmentation is applied.

    Args:
        dataframe:   DataFrame with columns ``nome`` and ``distancia``.
        image_dir:   Folder containing the image files.
        targets:     Pre-scaled target array aligned with ``dataframe``.
        target_size: (width, height) resize target.
        batch_size:  Number of samples per batch.
        shuffle:     Shuffle indices at the end of each epoch.
    """

    def __init__(
        self,
        dataframe: pd.DataFrame,
        image_dir,
        targets: np.ndarray,
        target_size: tuple = (224, 224),
        batch_size: int = 32,
        shuffle: bool = True,
    ):
        self.df          = dataframe.reset_index(drop=True)
        self.image_dir   = Path(image_dir)
        self.targets     = targets
        self.target_size = target_size
        self.batch_size  = batch_size
        self.shuffle     = shuffle
        self.indices     = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self) -> int:
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, batch_idx: int):
        start   = batch_idx * self.batch_size
        end     = min(start + self.batch_size, len(self.df))
        indices = self.indices[start:end]

        batch_imgs, batch_feats = [], []
        for idx in indices:
            img_bgr = cv2.imread(str(self.image_dir / self.df.loc[idx, "nome"]))
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            v       = extract_value_channel(img_rgb, self.target_size)
            feat    = extract_feature_vector(v)
            batch_imgs.append(v[..., np.newaxis])   # shape (H, W, 1)
            batch_feats.append(feat)

        X = {
            "image_input":   np.array(batch_imgs,  dtype=np.float32),
            "feature_input": np.array(batch_feats, dtype=np.float32),
        }
        y = self.targets[indices].astype(np.float32)
        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


# ── Split ──────────────────────────────────────────────────────────────────────
df_train, df_val, df_test = make_stratified_split(df, cfg)

# ── Scale targets ──────────────────────────────────────────────────────────────
# StandardScaler is fit only on training targets to prevent leakage.
scaler_y = StandardScaler()
y_train  = scaler_y.fit_transform(df_train[["distancia"]]).ravel()
y_val    = scaler_y.transform(df_val[["distancia"]]).ravel()
y_test   = scaler_y.transform(df_test[["distancia"]]).ravel()

print(f"\nTarget scaler — mean: {scaler_y.mean_[0]:.2f} cm  "
      f"std: {scaler_y.scale_[0]:.2f} cm")

# ── Instantiate sequences ──────────────────────────────────────────────────────
_size  = tuple(cfg["data"]["image_size"])
_batch = cfg["training"]["batch_size"]

train_seq = WaterDataSequence(df_train, IMAGE_DIR, y_train,
                              target_size=_size, batch_size=_batch, shuffle=True)
val_seq   = WaterDataSequence(df_val,   IMAGE_DIR, y_val,
                              target_size=_size, batch_size=_batch, shuffle=False)
test_seq  = WaterDataSequence(df_test,  IMAGE_DIR, y_test,
                              target_size=_size, batch_size=_batch, shuffle=False)

print(f"Batches — train: {len(train_seq)}  val: {len(val_seq)}  test: {len(test_seq)}")

# ── Inspect one batch ──────────────────────────────────────────────────────────
X_demo, y_demo = train_seq[0]
print(f"\nBatch shapes:")
print(f"  image_input   : {X_demo['image_input'].shape}   (B, H, W, 1)")
print(f"  feature_input : {X_demo['feature_input'].shape}   (B, 12)")
print(f"  targets       : {y_demo.shape}                       (B,)")

# Show a 4-image grid from the first batch
fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))
for i, ax in enumerate(axes):
    ax.imshow(X_demo["image_input"][i, :, :, 0], cmap="gray", vmin=0, vmax=1)
    alt_cm = scaler_y.inverse_transform([[y_demo[i]]])[0, 0]
    ax.set_title(f"true alt = {alt_cm:.0f} cm\n"
                 f"scaled y = {y_demo[i]:.3f}", fontsize=9)
    ax.axis("off")
fig.suptitle("First 4 images from training batch (V channel)", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# ── Cell 10: Custom Layers & Baseline CNN ─────────────────────────────────────
# Ported from waternet_v2/models/layers.py and waternet_v2/models/custom_cnn.py

import keras
from keras import layers as KL


class ClampedLinear(KL.Layer):
    """Linear activation with hard clamping to the valid altitude range.

    After the regression head produces an unconstrained float, this layer
    clips it to [min_val, max_val] so predictions never violate physical
    limits (50–800 cm in the normalised domain after inverse-scaling).

    Args:
        min_val: Lower clamp bound (same scale as model output).
        max_val: Upper clamp bound (same scale as model output).
    """

    def __init__(self, min_val: float = 0.0, max_val: float = 1.0, **kwargs):
        super().__init__(**kwargs)
        self.min_val = float(min_val)
        self.max_val = float(max_val)

    def call(self, inputs):
        return tf.clip_by_value(inputs, self.min_val, self.max_val)

    def get_config(self):
        config = super().get_config()
        config.update({"min_val": self.min_val, "max_val": self.max_val})
        return config


# ── ClampedLinear quick demo ───────────────────────────────────────────────────
clamp = ClampedLinear(min_val=0.0, max_val=1.0)
x_in  = tf.constant([-0.5, 0.0, 0.3, 0.7, 1.0, 1.5], dtype=tf.float32)
print("ClampedLinear demo  (min=0.0, max=1.0):")
print(f"  Input  : {x_in.numpy()}")
print(f"  Output : {clamp(x_in).numpy()}")
print("  Values outside [0, 1] are clipped — physically impossible altitudes rejected.\n")


def build_custom_cnn(
    input_shape: tuple = (224, 224, 1),
    conv_filters=None,
    dense_units=None,
    dropout_rate: float = 0.3,
    apply_physical_constraint: bool = False,
    alt_min_norm: float = 0.0,
    alt_max_norm: float = 1.0,
) -> keras.Model:
    """Build the custom VGG-inspired regression CNN (ablation baseline).

    Architecture:
        Input (224, 224, 1)
        → [Conv2D(f,5×5) → Conv2D(f,5×5) → MaxPool2D] × 4   (conv blocks)
        → GlobalAveragePooling2D
        → Dense(512) → Dropout → Dense(256) → Dropout → Dense(128) → Dropout
        → Dense(1, linear)   [→ ClampedLinear if apply_physical_constraint]

    This model accepts only the image branch (no feature vector input).
    It is the ablation baseline before multi-input fusion is introduced.

    Args:
        input_shape: (H, W, C) — expects (224, 224, 1) for the V channel.
        conv_filters: Filters per convolutional block. Default [32,64,128,256].
        dense_units:  Units in each dense regression layer. Default [512,256,128].
        dropout_rate: Dropout probability applied after deeper layers.
        apply_physical_constraint: If True, clamp output via ClampedLinear.
        alt_min_norm: Lower clamp bound.
        alt_max_norm: Upper clamp bound.

    Returns:
        Uncompiled tf.keras.Model.
    """
    if conv_filters is None:
        conv_filters = [32, 64, 128, 256]
    if dense_units is None:
        dense_units = [512, 256, 128]

    inp = keras.Input(shape=input_shape, name="image_input")
    x   = inp

    for i, n_filters in enumerate(conv_filters):
        x = KL.Conv2D(n_filters, (5, 5), padding="same",
                      activation="relu", name=f"conv_{i}_a")(x)
        x = KL.Conv2D(n_filters, (5, 5), padding="same",
                      activation="relu", name=f"conv_{i}_b")(x)
        x = KL.MaxPool2D(name=f"pool_{i}")(x)
        if i >= 2:
            x = KL.Dropout(dropout_rate, name=f"drop_conv_{i}")(x)

    x = KL.GlobalAveragePooling2D(name="gap")(x)

    for i, units in enumerate(dense_units):
        x = KL.Dense(units, activation="relu", name=f"dense_{i}")(x)
        x = KL.Dropout(dropout_rate, name=f"drop_dense_{i}")(x)

    output = KL.Dense(1, activation="linear", name="output")(x)

    if apply_physical_constraint:
        output = ClampedLinear(
            min_val=alt_min_norm, max_val=alt_max_norm,
            name="physical_constraint",
        )(output)

    return keras.Model(inputs=inp, outputs=output, name="WaterNet_CustomCNN")


# ── Instantiate and inspect ────────────────────────────────────────────────────
h, w = cfg["data"]["image_size"]
cnn_model = build_custom_cnn(
    input_shape=(h, w, 1),
    apply_physical_constraint=False,
)
cnn_model.summary()

total_p     = cnn_model.count_params()
trainable_p = sum(int(tf.size(w).numpy()) for w in cnn_model.trainable_weights)
print(f"\nTotal parameters     : {total_p:,}")
print(f"Trainable parameters : {trainable_p:,}")
print(f"Input shape          : {cnn_model.input_shape}")
print(f"Output shape         : {cnn_model.output_shape}")
print("\nNote: This is the ablation baseline (~1.2 M params).")
print("      ResNet50 transfer learning (~25.6 M params) follows in the next cell.")
